# HISEMOTIONS @ IberLEF 2026 — Análisis Exploratorio (EDA)

Objetivo: entender el dataset antes de entrenar cualquier modelo.
- Distribución de etiquetas en train vs dev
- Co-ocurrencias de emociones
- Longitud de los textos
- Ejemplos de textos por emoción

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

EMOTION_COLS = ['anger', 'fear', 'joy', 'sadness', 'surprise', 'hope']
COLORS = ['#e74c3c','#8e44ad','#f1c40f','#2980b9','#1abc9c','#e67e22']

In [ ]:
def load_split(path):
    df = pd.read_csv(path)
    df = df.dropna(subset=['text']).reset_index(drop=True)
    for col in EMOTION_COLS:
        if col in df.columns:
            df[col] = df[col].fillna(0).astype(int)
    return df

train_df = load_split('../train/train.csv')
dev_df   = load_split('../dev/dev.csv')

print(f'Train: {len(train_df)} rows | Dev: {len(dev_df)} rows')
train_df.head(3)

## 1. Distribución de etiquetas — Train vs Dev

In [ ]:
train_counts = train_df[EMOTION_COLS].sum()
dev_counts   = dev_df[EMOTION_COLS].sum()

dist = pd.DataFrame({
    'Train (count)': train_counts,
    'Train (%)':     (train_counts / len(train_df) * 100).round(1),
    'Dev (count)':   dev_counts,
    'Dev (%)':       (dev_counts / len(dev_df) * 100).round(1),
})
print(dist.to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, df_, title in zip(axes, [train_df, dev_df], ['Train', 'Dev']):
    counts = df_[EMOTION_COLS].sum()
    ax.bar(EMOTION_COLS, counts, color=COLORS)
    ax.set_title(f'{title} — label counts')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts):
        ax.text(i, v + 3, str(int(v)), ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 2. Fragmentos sin ninguna emoción (clase vacía)

In [ ]:
train_no_label = (train_df[EMOTION_COLS].sum(axis=1) == 0).sum()
dev_no_label   = (dev_df[EMOTION_COLS].sum(axis=1) == 0).sum()

print(f'Train sin etiqueta: {train_no_label} ({train_no_label/len(train_df)*100:.1f}%)')
print(f'Dev sin etiqueta:   {dev_no_label} ({dev_no_label/len(dev_df)*100:.1f}%)')

## 3. Número de etiquetas por fragmento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, df_, title in zip(axes, [train_df, dev_df], ['Train', 'Dev']):
    label_counts = df_[EMOTION_COLS].sum(axis=1)
    ax.hist(label_counts, bins=range(8), align='left', color='steelblue', edgecolor='white')
    ax.set_title(f'{title} — etiquetas por fragmento')
    ax.set_xlabel('Nº emociones')
    ax.set_ylabel('Fragmentos')
    ax.set_xticks(range(7))
plt.tight_layout()
plt.show()

## 4. Longitud de los textos (en tokens aprox.)

In [ ]:
train_len = train_df['text'].str.split().str.len()
dev_len   = dev_df['text'].str.split().str.len()

for name, lengths in [('Train', train_len), ('Dev', dev_len)]:
    print(f'{name}: media={lengths.mean():.1f} | mediana={lengths.median():.0f} | p95={lengths.quantile(0.95):.0f} | max={lengths.max()}')

plt.figure(figsize=(8, 4))
plt.hist(train_len, bins=50, alpha=0.6, label='Train', color='steelblue')
plt.hist(dev_len,   bins=50, alpha=0.6, label='Dev',   color='coral')
plt.axvline(256, color='red', linestyle='--', label='max_length=256')
plt.xlabel('Palabras')
plt.ylabel('Frecuencia')
plt.title('Distribución longitud de texto')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Co-ocurrencia de emociones (heatmap)

In [ ]:
combined = pd.concat([train_df, dev_df], ignore_index=True)
co_matrix = combined[EMOTION_COLS].T.dot(combined[EMOTION_COLS])

plt.figure(figsize=(7, 6))
sns.heatmap(co_matrix, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=EMOTION_COLS, yticklabels=EMOTION_COLS)
plt.title('Co-ocurrencia de emociones (train + dev)')
plt.tight_layout()
plt.show()

## 6. Ejemplos de texto por emoción

In [ ]:
for emo in EMOTION_COLS:
    samples = train_df[train_df[emo] == 1]['text'].head(2).tolist()
    print(f'\n=== {emo.upper()} ({int(train_df[emo].sum())} ejemplos en train) ===')
    for s in samples:
        print(f'  "{s[:200]}"')

## 7. Alerta: clases raras — anger y surprise

Con sólo 12 y 9 ejemplos de entrenamiento respectivamente, estas clases
necesitan tratamiento especial: `pos_weight` elevado en BCELoss, prompting LLM, o augmentación.

In [ ]:
for emo in ['anger', 'surprise']:
    n_train = int(train_df[emo].sum())
    n_dev   = int(dev_df[emo].sum())
    pos_w   = (len(train_df) - n_train) / max(n_train, 1)
    print(f'{emo}: train={n_train} | dev={n_dev} | pos_weight≈{pos_w:.1f}')